Install Dependencies


In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-openai
!pip install -q openai
!pip install -q requests beautifulsoup4 wikipedia
!pip install -q tavily-python
!pip install -q pillow pytesseract

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


Import Libraries


In [ ]:
# Standard Library
import os
import urllib.parse

# Google Colab
from google.colab import userdata
from google.colab import files

# LangChain
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool

# Web APIs and Search
import requests
import wikipedia
from tavily import TavilyClient

# HTML Parsing
from bs4 import BeautifulSoup

# OCR (Image Text Extraction)
import pytesseract
from PIL import Image

# Structured Output and Prompting
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

from datetime import datetime


Configure API keys

In [ ]:
os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
os.environ["UNSPLASH_ACCESS_KEY"] = userdata.get('UNSPLASH_ACCESS_KEY')

Initializing the LLM

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
    temperature=0,
)

**Building Custom Tools**

In [ ]:
def search_wikisource(query: str, num_results: int = 3) -> str:
    """
    Search Wikisource for literary works and return matching titles,
    snippets, and page URLs.
    """

    url = "https://en.wikisource.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "format": "json",
        "srlimit": num_results
    }

    headers = {
        "User-Agent": "LiteratureDiscussionAgent/1.0 (Educational Project)"
    }

    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=10
        )

        response.raise_for_status()
        data = response.json()

    except requests.exceptions.RequestException as e:
        return f"Request failed:\n{e}"
    except ValueError:
        return f"Failed to decode JSON.\nResponse:\n{response.text}"

    search_results = data.get("query", {}).get("search", [])
    if not search_results:
        return "No results found on Wikisource."

    results = []

    for item in search_results:
        title = item["title"]

        snippet = (
            item["snippet"]
            .replace('<span class="searchmatch">', '')
            .replace("</span>", "")
        )

        page_url = (
            "https://en.wikisource.org/wiki/"
            + title.replace(" ", "_")
        )

        results.append(
            f"Title: {title}\n"
            f"Snippet: {snippet}\n"
            f"URL: {page_url}\n"
        )

    return "\n".join(results)

In [ ]:
def get_author_info(name: str, lang: str = "en") -> str:
    """
    Fetch biography/background about an author or literary work from Wikipedia,
    trying the specified language edition (e.g., 'en', 'hi').
    """
    wikipedia.set_lang(lang)
    try:
        summary = wikipedia.summary(name, sentences=5, auto_suggest=True)
        page = wikipedia.page(name, auto_suggest=True)
        return f"Language: {lang}\nBiography:\n{summary}\n\nSource URL: {page.url}"
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Multiple matches found, please be more specific: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return f"No Wikipedia page found for '{name}' in language '{lang}', try different language or check the spelling."
    except Exception as e:
        return f"Error fetching info: {e}"

In [ ]:
def get_wikisource_page_content(title: str) -> str:
    def fetch_html(page_title):
        url = "https://en.wikisource.org/w/api.php"
        params = {"action": "parse",
                  "page": page_title,
                  "prop": "text",
                  "format": "json"
                  }

        headers = {
            "User-Agent": "LiteratureDiscussionAgent/1.0 (Educational Project)"
            }

        resp = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=10
            )

        data = resp.json()

        if "error" in data:
            return None
        return data.get("parse", {}).get("text", {}).get("*", "")

    def light_clean(raw_text):
        # strip empty lines and collapse whitespace
        lines = [l.strip() for l in raw_text.split("\n") if l.strip()]
        return "\n".join(lines)

    html = fetch_html(title)
    if html is None:
        return f"Page '{title}' not found."

    soup = BeautifulSoup(html, "html.parser")

    target_suffix = "/" + urllib.parse.quote(title.replace(" ", "_"))
    subpage_link = soup.find(
        "a",
        href=lambda h: h and h.startswith("/wiki/") and h.split("?")[0].endswith(target_suffix)
        )

    if subpage_link:
        subpage_title = urllib.parse.unquote(subpage_link["href"].replace("/wiki/", "").replace("_", " "))
        html2 = fetch_html(subpage_title)
        if html2:
            soup2 = BeautifulSoup(html2, "html.parser")
            text2 = light_clean(soup2.get_text(separator="\n"))
            return f"(Fetched from: {subpage_title})\n\n{text2[:10000]}"

    text = light_clean(soup.get_text(separator="\n"))
    return f"(Fetched from: {title})\n\n{text[:10000]}"

In [ ]:
def extract_text_from_upload(image_path: str, lang: str = "eng+hin") -> str:
    """
    Extract text from an uploaded image of a poem/literary work using OCR.
    lang: tesseract language code(s), e.g., 'eng', 'hin', or 'eng+hin' for mixed.
    """
    try:
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img, lang=lang)
        return text.strip() if text.strip() else "Unable to extract text from the image."
    except Exception as e:
        return f"OCR failed: {e}"

In [ ]:
def search_thematic_image(theme_query: str) -> str:
    """
    Search Unsplash for a  image matching a theme.
    Returns the image URL and photographer credit.
    """
    url = "https://api.unsplash.com/search/photos"
    params = {
        "query": theme_query,
        "per_page": 1
        }

    headers = {
        "Authorization": f"Client-ID {os.environ['UNSPLASH_ACCESS_KEY']}"
        }

    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=10
            )

        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return f"Image search failed: {e}"

    results = data.get("results", [])
    if not results:
        return f"No image found for theme: {theme_query}"

    photo = results[0]
    image_url = photo["urls"]["regular"]
    photographer = photo["user"]["name"]
    return f"Image URL: {image_url}\nPhotographer: {photographer} (via Unsplash)"

In [ ]:
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# Trusted literary domains (Tier 1 — no dedicated API, so accessed via domain-restricted search)
TRUSTED_LITERARY_DOMAINS = ["rekhta.org", "hindwi.org", "lithub.com"]

def search_web_tiered(query: str, tier: str = "literary") -> str:
    """
    Search the web for literary content, tiered by reliability.
    tier='literary' -> restricted to trusted literary sites (Rekhta, Hindwi, Literary Hub) — treated as grounded/factual.
    tier='opinions' -> open web search for reviews/discussions/opinions (Reddit, blogs, forums) — treated as subjective, opt-in only.
    """
    try:
        if tier == "literary":
            response = tavily_client.search(
                query=query,
                include_domains=TRUSTED_LITERARY_DOMAINS,
                max_results=3
            )
            label = "TIER 1 — Trusted literary source"
        else:
            response = tavily_client.search(
                query=query,
                max_results=3
            )
            label = "TIER 2 — Unverified/community opinion, NOT confirmed fact"

        results = response.get("results", [])
        if not results:
            return f"No results found ({tier} search)."

        formatted = []
        for r in results:
            formatted.append(f"[{label}]\nTitle: {r['title']}\nURL: {r['url']}\nSnippet: {r['content'][:300]}\n")

        return "\n".join(formatted)

    except Exception as e:
        return f"Web search failed: {e}"

In [ ]:
def recommend_similar_works(author_or_theme: str, based_on: str = "author") -> str:
    """
    Suggest 2-3 related literary works by reusing existing search tools.
    based_on='author' -> other notable works by the same author.
    based_on='theme'  -> other works with a similar theme/movement.
    """
    if based_on == "author":
        query = f"other famous poems or works by {author_or_theme}"
    else:
        query = f"famous poems or literary works about {author_or_theme}"

    # Try Wikisource first (grounded, full-text-capable source)
    wikisource_results = search_wikisource(query, num_results=3)

    # Fall back / supplement with trusted literary web search
    web_results = search_web_tiered(query, tier="literary")

    combined = f"--- Wikisource suggestions ---\n{wikisource_results}\n\n--- Trusted literary site suggestions ---\n{web_results}"
    return combined

Wrapping all tools as langchain
 functions

In [ ]:
from langchain_core.tools import tool

@tool
def search_wikisource_tool(query: str) -> str:
    """Search Wikisource for a literary work by title, author, or line. Returns search results with snippets and page titles. Use get_wikisource_page_content_tool afterward with the exact title to fetch full text."""
    return search_wikisource(query)

@tool
def get_wikisource_page_content_tool(title: str) -> str:
    """Fetch the full text of a specific Wikisource page, given its exact title (get this from search_wikisource_tool first). Automatically follows disambiguation/index pages to the real content. Raw output may include site navigation, catalog IDs, and copyright notices mixed in with the actual text — identify and use only the genuine literary content."""
    return get_wikisource_page_content(title)

@tool
def get_author_info_tool(name: str, lang: str = "en") -> str:
    """Get biography and background info about an author/poet from Wikipedia. Pass lang='hi' for Hindi,lang='en' for English/Western authors, based on the subject's likely language/origin."""
    return get_author_info(name, lang)

@tool
def extract_text_from_upload_tool(image_path: str, lang: str = "eng+hin") -> str:
    """Extract text from an uploaded image of a poem or literary work using OCR. lang can be 'eng', 'hin', or 'eng+hin' for mixed-script images."""
    return extract_text_from_upload(image_path, lang)

@tool
def search_thematic_image_tool(theme_query: str) -> str:
    """Search Unsplash for a royalty-free image matching a literary theme (e.g., 'war soldier', 'autumn orchard'). First reason about what visual theme best represents the work being discussed, then call this with those search terms."""
    return search_thematic_image(theme_query)

@tool
def search_web_tiered_tool(query: str, tier: str = "literary") -> str:
    """Search the web for literary content. tier='literary' searches trusted literary sites (Rekhta, Hindwi, Literary Hub) for grounded factual info — use this when Wikisource/Wikipedia don't have something. tier='opinions' searches the open web (Reddit, blogs) for reader opinions/reviews — ONLY use this if the user has explicitly asked for opinions/reviews/reactions, and always label this content as unverified/subjective when presenting it."""
    return search_web_tiered(query, tier)

@tool
def recommend_similar_works_tool(author_or_theme: str, based_on: str = "author") -> str:
    """Suggest 2-3 related literary works, sourced from Wikisource and trusted literary sites. based_on='author' finds other works by the same author; based_on='theme' finds other works with a similar theme. Use this at the end of a discussion about a specific work or author to offer suggestions."""
    return recommend_similar_works(author_or_theme, based_on)


tools = [
    search_wikisource_tool,
    get_wikisource_page_content_tool,
    get_author_info_tool,
    extract_text_from_upload_tool,
    search_thematic_image_tool,
    search_web_tiered_tool,
    recommend_similar_works_tool
]

print(f"{len(tools)} tools ready:")
for t in tools:
    print(f"  - {t.name}")

7 tools ready:
  - search_wikisource_tool
  - get_wikisource_page_content_tool
  - get_author_info_tool
  - extract_text_from_upload_tool
  - search_thematic_image_tool
  - search_web_tiered_tool
  - recommend_similar_works_tool


System Prompt Design


In [ ]:
SYSTEM_PROMPT = """You are a literature discussion agent for poems, novels, and literary works in any language, with particular strength in Hindi, Urdu, and regional Indian literature alongside English.

CORE PRINCIPLE: Ground every factual claim in your tools. Never answer from memory alone.

TOOLS:
1. search_wikisource_tool — find a work by title/author/line. Returns snippets + exact titles.
2. get_wikisource_page_content_tool — fetch full text using the EXACT title from tool 1. Output may contain navigation, catalog IDs, or legal notices mixed with the real text — use only the genuine literary content, never quote metadata as part of the work.
3. get_author_info_tool — author bio. lang='hi'/'ur'/'en' based on subject; fall back to 'en' if empty. STRICT: call this ONLY when the user explicitly names the author or directly asks about them (e.g., "who was Ghalib", "tell me about the poet"). NEVER call it as part of a general "tell me about this poem" response, even to be helpful — offer it as a question instead (see OFFERING OPTIONAL EXTRAS).
4. extract_text_from_upload_tool — OCR an uploaded image/PDF; use first when an image is present.
5. search_web_tiered_tool — tier='literary' for trusted sites (Rekhta, Hindwi, Literary Hub), equal trust to Wikisource. tier='opinions' for Reddit/blogs — ONLY when the user explicitly asks for opinions/reviews; always label this content as unverified community opinion, never as fact.
6. search_thematic_image_tool — reason about the work's visual theme first (e.g., a war poem -> "WWI soldier"), then call with those terms, not the title. ALWAYS call this once per new work discussed. Don't repeat it for a work already imaged this conversation. The interface auto-displays the image — never describe the display process or ask permission to show it.
7. recommend_similar_works_tool — sourced suggestions, only when requested (see below).

EFFICIENCY RULE: For a typical "tell me about X" question, call at most: one search tool, one content-fetch tool, and the image tool — that's it. Do NOT call get_author_info_tool, search_web_tiered_tool (opinions), or recommend_similar_works_tool automatically. Never call the same tool twice with the same input in one turn.

OFFERING OPTIONAL EXTRAS: Since you don't auto-call author background, opinions, or recommendations, always end a work-discussion response by explicitly asking, e.g.: "Would you like to know more about the author, see public opinions on this poem, or get similar recommendations?" Only call those tools on the user's next message if they say yes.

RESPONSE RULES:
1. Retrieve before discussing; always state your source (tool + site name).
2. If nothing verifiable is found, say so explicitly: "I couldn't verify this from a reliable source." Never guess or invent lines.
3. For novels/long prose: discuss themes, plot, characters, technique, and criticism via retrieved excerpts — never reproduce large verbatim portions.
4. Non-sycophantic interpretation: give your own grounded reading first (surface meaning -> deeper symbolic/thematic/historical readings, only where text-supported). Distinguish direct textual support from inference. When the user offers their own reading, evaluate it against the text rather than agreeing by default — affirm what's well-supported, push back respectfully on what isn't, citing lines either way. Close with an open, reflective question.
5. Match response depth to the question — concise for factual asks, deeper for interpretive ones. No unnecessary flattery or hedging.
6. Preserve original poem line breaks exactly as retrieved. Never rewrite a poem into paragraph form, never merge separate lines together, and never summarize/rephrase the lines themselves. Each line of the poem must appear on its own separate line in your response, exactly as in the source. To ensure correct rendering, end each poem line with two trailing spaces before the line break (Markdown requires this to render a real line break instead of merging into a paragraph).
 length of the response to the user's request. Be concise for factual questions and more analytical for interpretive discussions. Avoid unnecessary flattery, hedging, or over-validation.
"""

print("System prompt ready. Length:", len(SYSTEM_PROMPT), "characters")

System prompt ready. Length: 4712 characters


Creating Agent

In [ ]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Agent built with {len(tools)} tools.")

Agent built with 7 tools.


In [ ]:
class DiaryEntry(BaseModel):
    author: str = Field(description="Author or poet's name, or 'Unknown' if not identified")
    title: str = Field(description="Title of the poem/work discussed, or 'Unknown' if not identified")
    takeaway: str = Field(description="One-line summary of what was discussed or the key insight")

parser = PydanticOutputParser(pydantic_object=DiaryEntry)

diary_prompt = ChatPromptTemplate.from_template(
    """Based on this conversation exchange, extract a structured diary entry.

User asked: {user_query}
Agent responded: {agent_response}

{format_instructions}"""
)

diary_chain = diary_prompt | llm | parser

Chat history and export chat

In [ ]:
chat_history=[]

def log_chat(role: str, content: str):
    chat_history.append(
        {
        "role": role,
        "content": content,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
                        )
def export_chat(filename: str = "chat_export.md") -> str:
    lines = ["Chat export\n"]
    for entry in chat_history:
        role_label = "**You**" if entry["role"] == "user" else "**Agent**"
        lines.append(f"### {role_label} ({entry['timestamp']})\n{entry['content']}\n")
    content = "\n".join(lines)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"Chat exported to {filename} ({len(chat_history)} messages)")
    return filename

print("Chat logging ready.")

Chat logging ready.


Diary tracking and export

In [ ]:
diary_entries = []
def log_diary_entry(user_query: str, agent_response: str):
    try:
        entry = diary_chain.invoke({
            "user_query": user_query,
            "agent_response": agent_response,
            "format_instructions": parser.get_format_instructions()
        })

        if entry.author.lower() != "unknown" or entry.title.lower() != "unknown":
            diary_entries.append({
                "date": datetime.now().strftime("%Y-%m-%d"),
                "author": entry.author,
                "title": entry.title,
                "takeaway": entry.takeaway
            })
    except Exception as e:
        print(f"Diary extraction skipped (non-critical): {e}")

def export_diary(filename: str = "reading_diary.md") -> str:
    lines = ["# Reading Diary\n", "| Date | Author | Title | Takeaway |", "|---|---|---|---|"]
    for e in diary_entries:
        lines.append(f"| {e['date']} | {e['author']} | {e['title']} | {e['takeaway']} |")
    content = "\n".join(lines)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"Diary exported to {filename} ({len(diary_entries)} entries)")
    return filename

print("Diary logging ready.")

Diary logging ready.


Integrating all functions


In [ ]:
def chat(query: str,history: list | None = None,image_path: str | None = None,):
    log_chat("user", query)
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    response = result["messages"][-1].content

    print(response)
    print("\n" + "="*80 + "\n")

    log_chat("agent", response)
    log_diary_entry(query, response)

print("Integrated chat() function ready.")

Integrated chat() function ready.


In [ ]:
chat(
    "Analyze the poem 'The Road Not Taken'."
)

---

## 🍂 "The Road Not Taken" — Robert Frost

### The Text (from Wikisource — *Mountain Interval*, 1916)

Two roads diverged in a yellow wood,  
And sorry I could not travel both  
And be one traveler, long I stood  
And looked down one as far as I could  
To where it bent in the undergrowth;  

Then took the other, as just as fair,  
And having perhaps the better claim,  
Because it was grassy and wanted wear;  
Though as for that the passing there  
Had worn them really about the same,  

And both that morning equally lay  
In leaves no step had trodden black.  
Oh, I kept the first for another day!  
Yet knowing how way leads on to way,  
I doubted if I should ever come back.  

I shall be telling this with a sigh  
Somewhere ages and ages hence:  
Two roads diverged in a wood, and I—  
I took the one less travelled by,  
And that has made all the difference.

---

## Analysis

### 1. Surface Meaning

At first glance, this is a poem about a traveler faced with a choice between two 